In [1]:
from graphviz import Digraph

In [2]:
dot = Digraph("Flow", format="png")
dot.attr(rankdir="TB", size="8")

# Start
dot.node("start", "main.f90", shape="oval", style="filled", color="red")

# Main decisions
dot.node("read", "Read run_jet_calc\nrun_sed_calc", shape="parallelogram")
dot.node("jet?", "Is run_jet_calc true?", shape="diamond")
#dot.node("sed?", "Is run_sed_calc true?", shape="diamond") Is run_sed_calc test run?

#Outside f90 files
dot.node("controls", "controls.f90", shape="oval", style="filled", color='yellow')
dot.node("initialize", "initialize.f90", shape="oval", style="filled", color='yellow')
dot.node("sedsyn", "synrad.f90", shape="oval", style="filled", color='yellow')
dot.node("quantities", "evolution.f90", shape="oval", style="filled", color='yellow')
dot.node("sedic", "icrad.f90", shape="oval", style="filled", color='yellow')
dot.node("sedssc", "sscrad.f90", shape="oval", style="filled", color='yellow')
dot.node("edist", "edistribution.f90", shape="oval", style="filled", color='yellow')

# Jet branch
dot.node("runjet", "Call run_jet", shape="box")
dot.node("traj", "Call trajectory_calc", shape="box")
dot.node("quant", "Call Quantities", shape="box")
dot.node("partE", "Calculate Particle\nEnergy Distribution", shape="box")

# SED branch
dot.node("synch", "Calculate SED Synchrotron", shape="box")
dot.node("ic", "Calculate SED IC", shape="box")
dot.node("ssc", "Calculate SED SSC", shape="box")

# Flow connections
dot.edge("controls", "traj",'JET_\nCJ_', style="dashed")
dot.edge("initialize", "traj",'allocate vectors', style="dashed")
dot.edge("start", "read")
dot.edge('controls','read','run_jet_calc', style="dashed")
dot.edge('controls','partE','NE_calc\n Np_calc', style="dashed")
dot.edge('controls','synch','radsyn_calc\n radpsyn_calc', style="dashed")
dot.edge('controls','ic','radic_calc', style="dashed")
dot.edge('controls','ssc','radssc_calc', style="dashed")
dot.edge("read", "jet?")
dot.edge("jet?", "runjet", label="yes")
#dot.edge("jet?", "sed?", label="test run")  Comment unless sed is not test

dot.edge("runjet", "traj")
dot.edge("traj", "quant")
dot.edge("quant", "partE")
dot.edge("quant", "quantities")

dot.edge("partE", "synch", label="yes")
dot.edge("partE", "edist")
dot.edge("synch", "ic")
dot.edge("ic", "ssc")

dot.edge("ssc", "sedssc")
dot.edge("synch", "sedsyn")
dot.edge("ic", "sedic")


# Output final image
dot.render("program_flow", view=True, cleanup=True)


'program_flow.png'

/snap/core20/current/lib/x86_64-linux-gnu/libstdc++.so.6: version `GLIBCXX_3.4.29' not found (required by /lib/x86_64-linux-gnu/libproxy.so.1)
Failed to load module: /home/tomas/snap/code/common/.cache/gio-modules/libgiolibproxy.so
eog: symbol lookup error: /snap/core20/current/lib/x86_64-linux-gnu/libpthread.so.0: undefined symbol: __libc_pthread_init, version GLIBC_PRIVATE


In [ ]:

dot.clear()
dot = Digraph("EDistribution_Flow", format="png")
dot.attr(rankdir="TB", size="8")

# Start
dot.node("start", "edistribution.f90", shape="oval", style="filled", color="red")

# Main subroutines
dot.node("edist", "Edist", shape="box")
#dot.node("calcne_onezone", "calcNE_onezone", shape="box")

#dot.node("fp_findif_cool", "FP_FinDif_cool", shape="box")
dot.node("n_bin", "N_bin\n(bins and integrates the particle \nenergy distribution ( N(E, z) ) over different\n spatial (z) ranges along the jet and \noutputs the results to files.)", shape="box")
dot.node("calc_ue", "Calc_Ue", shape="box")
dot.node("f90", "losses.f90", shape="oval",style='dashed, filled', color='yellow')

dot.node('quant_cond', 'print_quantities', shape='diamond', width="0.1")

# Helper functions (rank same)
with dot.subgraph() as sub:
    sub.attr(rank='same')
    sub.node("calcne_paramo", "calcNE_paramo", shape="box")
    sub.node("helpers1", "Helper Functions:\nCutE, BreakE, qp_E, FP_FinDif_cool", 
             shape="note", color="green")

#Output files
with dot.subgraph(name="output files") as c:
    c.attr(label='output', style="rounded, filled")
    c.node_attr.update(style="rounded, filled", color="lightgrey")
    c.node("entry", "Output files", shape="oval", width="0.1")
    c.node("N(E)", "N//particle//_cuts.dat\n\n(E and N(E) for each jet cut)", shape="box")
    c.node("quantities", "Quantities.dat", shape="box")
    c.node('z','z_bin.dat\n\nz-bin edges',shape='box')
    c.node('U_e','U_e.dat',shape='box')
    c.node('N_bin','N//particle//_bin.dat\n\nN(E) binned in each z-range',shape='box')
    c.node('N_total','N//particle//_total.dat\n\nTotal N(E) integrated over all z',shape ='box')
    c.node("E", "E//particle//_max_FF.dat\n\n(Cutoff energy for particle\n inyection per segment)", shape="box")
    c.edges([("entry", "N(E)"), ("entry", "quant_cond"), ("entry", "E")])


# Flow connections
dot.edge('quant_cond', 'quantities','yes')
dot.edge('start','edist')
dot.edge('edist', 'entry')
dot.edge("edist", "calcne_paramo",'E(:,z),\npart')
dot.edge('f90','calcne_paramo','Import functions')
dot.edge("calcne_paramo", "edist",'N(E,z)')
dot.edge("edist", "n_bin", 'E(:,z),\npart,\nN(E,z)')
dot.edge("n_bin", "N_bin")
dot.edge("n_bin", "N_total")
dot.edge("n_bin", "z")
dot.edge('calc_ue','U_e')


dot.edge("calcne_paramo", "helpers1", style="dashed")

# Output
dot.render("edistribution_flow", view=True, cleanup=True)


'edistribution_flow.png'

/snap/core20/current/lib/x86_64-linux-gnu/libstdc++.so.6: version `GLIBCXX_3.4.29' not found (required by /lib/x86_64-linux-gnu/libproxy.so.1)
Failed to load module: /home/tomas/snap/code/common/.cache/gio-modules/libgiolibproxy.so
eog: symbol lookup error: /snap/core20/current/lib/x86_64-linux-gnu/libpthread.so.0: undefined symbol: __libc_pthread_init, version GLIBC_PRIVATE


In [ ]:
from graphviz import Digraph

dot = Digraph("Evolution_Flow", format="png")
dot.attr(rankdir="TB", size="8")

# Start node (file)
dot.node("start", "evolution.f90", shape="oval", style="filled", color="red")

# Main subroutine
dot.node("quantities", "Quantities", shape="box")
dot.node("controls", "controls.f90", shape="oval",style='dashed, filled', color='yellow')
dot.node("globals", "globals.f90\nsystem_parameters.f90", shape="oval",style='dashed, filled', color='yellow')


# Input
dot.node("input", "jet ('' or 'CJ')", shape="parallelogram", color="lightblue")
dot.node('import', 'Imported parameters and values\nfrom globals.f90 and\n system_parameters.f90\n(Ns, jo, etc.)', shape="box", style='filled', color='lightblue')

# Initialization
dot.node("init", "Allocate joFF(Ns)\nOpen output files", shape="box")

# Direction selection
dot.node("dir", "Set dir\n(based on jet type)", shape="diamond")

# Main loop
dot.node("loop", "For i = 1 to Ns:\n- Compute jet segment properties\n- Write to files", shape="box")

# Output files
dot.node("out1", "quant_FF{jet}.dat", shape="box", color="lightgrey")
dot.node("out2", "quant_FF_norm{jet}.dat", shape="box", color="lightgrey")

# End
dot.node("end", "End\nDeallocate joFF", shape="oval", style="filled", color="green")

# Connections
dot.edge("start", "quantities")
dot.edge('controls', 'input')
dot.edge("input", "quantities")
dot.edge('globals', 'import')
dot.edge("import", "init")
dot.edge("quantities", "init")
dot.edge("init", "dir")
dot.edge("dir", "loop")
dot.edge("loop", "out1")
dot.edge("loop", "out2")
dot.edge("loop", "end")

# Render
dot.render("evolution_flow", view=True, cleanup=True)

'evolution_flow.png'

/snap/core20/current/lib/x86_64-linux-gnu/libstdc++.so.6: version `GLIBCXX_3.4.29' not found (required by /lib/x86_64-linux-gnu/libproxy.so.1)
Failed to load module: /home/tomas/snap/code/common/.cache/gio-modules/libgiolibproxy.so


eog: symbol lookup error: /snap/core20/current/lib/x86_64-linux-gnu/libpthread.so.0: undefined symbol: __libc_pthread_init, version GLIBC_PRIVATE


In [ ]:
from graphviz import Digraph

dot = Digraph("Trajectory_Flow", format="png")
dot.attr(rankdir="TB", size="8")

# Start node (file)
dot.node("start", "trajectory.f90", shape="oval", style="filled", color="red")
dot.node("controls", "controls.f90", shape="oval",style='dashed, filled', color='yellow')

# Main subroutine
dot.node("traj", "trajectory_calc", shape="box")

dot.node('sys_par','system_parameters.f90',shape='oval',style='dashed, filled', color='yellow')
dot.node('globals','globals.f90',shape='oval',style='dashed, filled', color='yellow')

# Input
dot.node("input", "jet ('' or 'CJ')", shape="parallelogram", color="lightblue")

# Direction selection
dot.node("dir", "Set dir\n(based on jet type)", shape="diamond")

# Initialization
dot.node("init", "z(1) = dir*z0\nrj_out(1) = rout_0\nrj_in(1) = rin_0", shape="box")

# Main loop
dot.node("loop", "Do loops over z bins", shape="box")

# Open file
dot.node("open", "trajectory{jet}.dat", shape="box")

# Write loop
dot.node("write", "For i = 1 to Ns:\nWrite z(i), rj_out(i), rj_in(i)", shape="box")


# End
dot.node("end", "End", shape="oval", style="filled", color="green")

# Connections
dot.edge("start", "traj")
dot.edge("input", "traj")
dot.edge("traj", "dir")
dot.edge('controls', 'input')
dot.edge("dir", "init")
dot.edge('sys_par', 'init','z0\nrout_0\nrin_0')
dot.edge('sys_par', 'loop','r_fun_out\nr_fun_in')
dot.edge("init", "loop")
dot.edge("globals", "loop", 'variables')
dot.edge("loop", "write")
dot.edge("write", "open")
dot.edge("write", "end")

# Render
dot.render("trajectory_flow", view=True, cleanup=True)

'trajectory_flow.png'

/snap/core20/current/lib/x86_64-linux-gnu/libstdc++.so.6: version `GLIBCXX_3.4.29' not found (required by /lib/x86_64-linux-gnu/libproxy.so.1)
Failed to load module: /home/tomas/snap/code/common/.cache/gio-modules/libgiolibproxy.so
eog: symbol lookup error: /snap/core20/current/lib/x86_64-linux-gnu/libpthread.so.0: undefined symbol: __libc_pthread_init, version GLIBC_PRIVATE


In [ ]:
from graphviz import Digraph

dot = Digraph("Synrad_Flow", format="png")
dot.attr(rankdir="TB", size="8")

# Start node
dot.node("start", "synrad.f90", shape="oval", style="filled", color="red")

# Main subroutines
dot.node("sedsyn_bin", "SEDsyn_bin", shape="box")
with dot.subgraph() as same_rank:
    same_rank.attr(rank='same')
    same_rank.node("sedsyn", "SEDsyn", shape="box")
    same_rank.node('Flux_loop','loop over z bins and \nvalues of energy vector', shape='box')
dot.node('dFlux_dz','dFlux_dz\nFlux at a differential z', shape='box')
with dot.subgraph() as subgraph:
    subgraph.attr(rank='same')
    subgraph.node('j','j_sync\nSynchrotron emissivity', shape='box')
    subgraph.node('alpha_sync','alpha_sync\nSynchrotron absorption coefficient', shape='box')
    subgraph.node('crossed_length','crossed_length\nLength of the crossed region', shape='box')
dot.node('PSynchrotron','PSynchrotron\nSynchrotron emitted power', shape='box')


# File I/O
dot.node('bin_SEDsyn', 'bin_SEDsyn', shape='diamond')
dot.node("file_in", "Read E(:,z),N(E,z) from file", shape="parallelogram", color="blue", width="0.1")
dot.node('file_out_1', "dFlux_dz//jet//.dat\nValues of Flux, \nB, R_in, R_out, etc.\nfor any given z",shape='parallelogram', color='blue', width="0.1")
dot.node('file_out_2', "Flux//jet//.dat\nContribution of the \nsegments up to given \nlengths",shape='parallelogram', color='blue')


# Connections
dot.edge("start", "sedsyn")
dot.edge("file_in","sedsyn")
dot.edge("sedsyn", "Flux_loop", xlabel='N(E,z), \nE(:,z),\n B(z),\n R_in(z),\n R_out(z),\n delta(z)')
dot.edge("Flux_loop", "dFlux_dz")
dot.edge("dFlux_dz","Flux_loop")
dot.edge('Flux_loop','file_out_1')
dot.edge('Flux_loop','file_out_2')
dot.edge("dFlux_dz", "j")
dot.edge("dFlux_dz", "alpha_sync")
dot.edge("dFlux_dz", "crossed_length")
dot.edge("j", "PSynchrotron")
dot.edge("PSynchrotron", "j")
dot.edge("alpha_sync", "PSynchrotron")
dot.edge("PSynchrotron", "alpha_sync")
dot.edge("sedsyn", 'bin_SEDsyn')
dot.edge('bin_SEDsyn',"sedsyn_bin")

# Render
dot.render("synrad_flow", view=True, cleanup=True)

'synrad_flow.png'

/snap/core20/current/lib/x86_64-linux-gnu/libstdc++.so.6: version `GLIBCXX_3.4.29' not found (required by /lib/x86_64-linux-gnu/libproxy.so.1)
Failed to load module: /home/tomas/snap/code/common/.cache/gio-modules/libgiolibproxy.so
eog: symbol lookup error: /snap/core20/current/lib/x86_64-linux-gnu/libpthread.so.0: undefined symbol: __libc_pthread_init, version GLIBC_PRIVATE


In [ ]:
from graphviz import Digraph

dot = Digraph("ProgramFlow", format="png")
dot.attr(rankdir="TB", size="10,12")

# Main entry
dot.node("main", "main.f90", shape="ellipse", style="filled", color="red")

# Controls and initialization
dot.node("controls", "controls.f90\n(User Parameters)", shape="ellipse", style="filled", color="yellow")
dot.node("init", "initialize.f90\n(Allocate vectors, setup)", shape="box")
dot.edge("main", "controls")
dot.edge("main", "init")

# Jet trajectory
dot.node("traj", "trajectory.f90\nJet Geometry\n(z, Rin, Rout)", shape="box")
dot.edge("init", "traj")
dot.edge("controls", "traj")

dot.node("sys", "system_parameters.f90\n(z0, Rin0, Rout0)", shape="ellipse", style="filled", color="yellow")
dot.node("glob", "globals.f90\n(shared variables)", shape="ellipse", style="filled", color="yellow")
dot.edge("sys", "traj")
dot.edge("glob", "traj")

# Evolution of jet quantities
dot.node("evol", "evolution.f90\n(Physical Quantities)", shape="box")
dot.edge("traj", "evol")
dot.edge("controls", "evol")
dot.edge("glob", "evol")

# Energy distribution
dot.node("edist", "edistribution.f90\nParticle Energy Distribution ", shape="box")
dot.node("losses", "losses.f90\nCooling & Losses", shape="ellipse", style="filled", color="yellow")
dot.edge("evol", "edist")
dot.edge("losses", "edist")

# Radiation modules
dot.node("syn", "synrad.f90\nSynchrotron", shape="box")
dot.node("ic", "icrad.f90\nInverse Compton", shape="box")
dot.node("ssc", "sscrad.f90\nSynchrotron Self-Compton", shape="box")

dot.edge("edist", "syn")
dot.edge("edist", "ic")
dot.edge("edist", "ssc")
dot.edge("controls", "syn")
dot.edge("controls", "ic")
dot.edge("controls", "ssc")

# Outputs
dot.node("out", "Output Files:\n- trajectory{jet}.dat\n- Quantities.dat\n- N(E).dat\n- Flux_syn.dat\n- Flux_IC.dat\n- Flux_SSC.dat",
         shape="note", color="blue")

dot.edge("traj", "out")
dot.edge("evol", "out")
dot.edge("edist", "out")
dot.edge("syn", "out")
dot.edge("ic", "out")
dot.edge("ssc", "out")

# End
dot.node("end", "End", shape="ellipse", style="filled", color="green")
dot.edge("out", "end")

dot.render("program_flow_detailed", cleanup=True)



'program_flow_detailed.png'

In [7]:
from graphviz import Digraph

dot = Digraph("DualFlow", format="png")
dot.attr(rankdir="TB", size="10,14")

# Inputs







with dot.subgraph() as s:
    s.attr(rank='same')
    s.node("inputs", 
           "Inputs & Initialization\n- Jet type (jet/CJ)\n- z₀, Rin₀, Rout₀\n- Energy grid", 
           shape="parallelogram", style="filled", color="lightyellow")
    s.node("main", "main.f90", shape="oval",style='dashed, filled', color='yellow')
    s.node("controls", "controls.f90", shape="oval",style='dashed, filled', color='yellow')
    s.node("initialize", "initialize.f90", shape="oval",style='dashed, filled', color='yellow')

with dot.subgraph() as same_rank_geom:
    same_rank_geom.attr(rank='same')
    same_rank_geom.node("geom", 
         "Jet Geometry (Trajectory)\n- z bins\n- Rin(z), Rout(z)\n- Direction", 
         shape="box", style="filled", color="lightblue")
    same_rank_geom.node("trayectory", "trayectory.f90", shape="oval",style='dashed, filled', color='yellow')
    same_rank_geom.node("sys_par", "system_parameters.f90", shape="oval",style='dashed, filled', color='yellow')
    same_rank_geom.node(
        "geom_desc",
        "calculates a vertical jet trajectory\noutput: trayectory//jet//.dat\ncontains: z(i), rin(i), rout(i)",
        shape="note",
        color="lightblue"
    )


# Evolution
with dot.subgraph() as same_rank_evol:
    same_rank_evol.attr(rank='same')
    same_rank_evol.node("evol", 
         "Jet Evolution\n- B(z)\n- δ(z)\n- Densities, jet power", 
         shape="box", style="filled", color="lightblue")
    same_rank_evol.node("evolution", "evolution.f90", shape="oval",style='dashed, filled', color='yellow')
    same_rank_evol.node(
        "evol_desc",
        "calculates the evolution of different quantities from cell to cell.\noutput: quant_FF//jet//.dat, quant_FF_norm//jet//.dat.",
        shape="note",
        color="lightblue"
    )

# Particle distribution
with dot.subgraph() as same_rank_dist:
    same_rank_dist.attr(rank='same')
    same_rank_dist.node("dist", 
         "Particle Energy Distribution\n- N(E,z)\n- Acceleration\n- Synch/IC/Adiabatic losses", 
         shape="box", style="filled", color="lightgreen")
    same_rank_dist.node("edistribution", "edistribution.f90", shape="oval",style='dashed, filled', color='yellow')
    same_rank_dist.node("losses", "losses.f90", shape="oval",style='dashed, filled', color='yellow')
    same_rank_dist.node(
        "dist_desc",
        "calculates the electron energy distribution at each segment.\noutput: E'//particle//'_max_FF.dat, N'//particle//'_cuts.dat\ncontains: E and N(E) for each jet cut",
        shape="note",
        color="lightgreen"
    )

# Radiation processes
with dot.subgraph() as same_rank:
    same_rank.attr(rank='same')
    same_rank.node("rad", "Radiation Processes", shape="diamond", style="filled", color="lightpink")
    same_rank.node("synrad", "synrad.f90", shape="oval",style='dashed, filled', color='yellow')
    same_rank.node("icrad", "icrad.f90", shape="oval",style='dashed, filled', color='yellow')
    same_rank.node("sscrad", "sscrad.f90", shape="oval",style='dashed, filled', color='yellow')
with dot.subgraph() as same_rank_syn:
    same_rank_syn.node("syn", "Synchrotron\n(e⁻ in B(z))", shape="ellipse")
    same_rank_syn.node(
    "syn_desc",
    "calculates the synchrotron SED.\nOutputs: dFlux_dz\"//jet//\".dat, Flux'//jet//'.dat",
    shape="note",
    color="lightpink")
dot.node("ic", "Inverse Compton\n(external photons)", shape="ellipse")
dot.node("ssc", "SSC\n(synchrotron photons)", shape="ellipse")


# SED
dot.node("sed", 
         "SED Construction\n- Integrate over z bins\n- Syn + IC + SSC", 
         shape="box", style="filled", color="lightgrey")

# Outputs
dot.node("out", 
         "Outputs\n- Trajectory.dat\n- Quantities.dat\n- N(E,z)\n- Flux_syn/IC/SSC\n- Total SED", 
         shape="note", color="white")

# Edges

# For each rank, add edges from every other node to the first node in that rank

# Inputs rank
# s contains: inputs, main, controls, initialize
dot.edge("main", "inputs")
dot.edge("controls", "inputs")
dot.edge("initialize", "inputs")

# Geometry rank
# same_rank_geom contains: geom, trayectory
dot.edge("trayectory", "geom")
dot.edge("sys_par", "geom")
dot.edge('geom','geom_desc')

# Evolution rank
# same_rank_evol contains: evol, evolution
dot.edge("evolution", "evol")
dot.edge('evol','evol_desc')

# Distribution rank
# same_rank_dist contains: dist, edistribution
dot.edge("edistribution", "dist")
dot.edge("losses", "dist")
dot.edge('dist','dist_desc')

# Radiation rank
# same_rank contains: rad, synrad, icrad, sscrad
dot.edge('synrad','syn_desc')
dot.edge("synrad", "rad")
dot.edge("icrad", "rad")
dot.edge("sscrad", "rad")
dot.edge("inputs", "geom")
dot.edge("geom", "evol")
dot.edge("evol", "dist")
dot.edge("dist", "rad")
dot.edge("rad", "syn")
dot.edge("rad", "ic")
dot.edge("rad", "ssc")
dot.edge("syn", "sed")
dot.edge("ic", "sed")
dot.edge("ssc", "sed")
dot.edge("sed", "out")



dot.render("conceptual_program_flow", cleanup=True)


'conceptual_program_flow.png'